<h1 style="color:red">Column Transformer</h1>

In [1]:
import pandas as pd 
import numpy as np 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [2]:
df = pd.read_csv("covid_toy.csv")
df.sample(5)         
# age, fewer   -> numerical column        # fewer has some missing values
# gender, city -> nominal data 
#     cough    -> ordinal data    (Mild < Strong)

,age,gender,fever,cough,city,has_covid
7,20,Female,NaN,Strong,Mumbai,Yes
94,79,Male,NaN,Strong,Kolkata,Yes
39,50,Female,103.0,Mild,Kolkata,No
58,23,Male,98.0,Strong,Mumbai,Yes
27,33,Female,102.0,Strong,Delhi,No


In [3]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [4]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [5]:
df.isnull().sum()       # fewer has 10 missing values
#  we will imply SimpleImputer to handle missing values

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

<h3 style="color:brown">Train Test Split</h3>

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df['has_covid'], test_size=0.25, random_state=0) 

X_train.shape, X_test.shape

((75, 5), (25, 5))

In [7]:
X_train.head()

,age,gender,fever,cough,city
48,66,Male,99.0,Strong,Bangalore
6,14,Male,101.0,Strong,Bangalore
99,10,Female,98.0,Strong,Kolkata
82,24,Male,98.0,Mild,Kolkata
76,80,Male,100.0,Mild,Bangalore


<h2 style="color:purple">1). Aam Zindagi</h2>

In [22]:
# impute missing values of fewer column 
si = SimpleImputer()
# adding simple imputer to fewer column 
X_train_fever = si.fit_transform(X_train[['fever']])      # all missing values has been replaced by the mean

X_test_fever = si.transform(X_test[['fever']])      # also transforming test data

X_train_fever = np.round(X_train_fever, decimals=2)
X_test_fever = np.round(X_test_fever, decimals=2)
display(X_train_fever.shape)
X_train_fever[:13]            # 12th index row has missing value, which is replaced by it's mean, (0-based indexing)

(75, 1)

array([[ 99.  ],
       [101.  ],
       [ 98.  ],
       [ 98.  ],
       [100.  ],
       [102.  ],
       [ 99.  ],
       [ 99.  ],
       [104.  ],
       [100.  ],
       [102.  ],
       [ 98.  ],
       [100.92]])

In [14]:
# Ordinal Encoding  -> cough
oe = OrdinalEncoder(categories= [['Mild','Strong']], dtype=np.int32)

X_train_cough = oe.fit_transform(X_train[['cough']])    # training and transforming training data

X_test_cough = oe.transform(X_test[['cough']])          # transforming test data

display(X_train_cough.shape)
X_train_cough[:5]

(75, 1)

array([[1],
       [1],
       [1],
       [0],
       [0]], dtype=int32)

In [15]:
# OneHotEncoding on gender, city 
ohe = OneHotEncoder(drop='first', dtype=int, sparse_output=False)

In [16]:
X_train_ohe = ohe.fit_transform(X_train[['gender','city']])
X_test_ohe = ohe.transform(X_test[['gender','city']])

display(X_train_ohe.shape) 
X_train_ohe[:5]

(75, 4)

array([[1, 0, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 1, 0],
       [1, 0, 1, 0],
       [1, 0, 0, 0]])

In [24]:
#  convert all numpy arrays into seperate dataframes and later on merge them

# X_train_fever = pd.DataFrame(np.round(X_train_fever, decimals=2), columns=['fever'], index=X_train.index)
# X_test_fever = pd.DataFrame(np.round(X_test_fewer, decimals=2), columns=['fever'], index=X_test.index)

# X_train_cough = pd.DataFrame(X_train_cough, columns=['cough'], index=X_train.index)
# X_test_cough = pd.DataFrame(X_test_cough, columns=['cough'], index=X_test.index)

# X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(),index=X_train.index)
# X_test_ohe = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(),index=X_test.index) 

# # concatenate
# X_train_new = pd.concat([X_train['age'], X_train_fever, X_train_cough, X_train_ohe], axis=1)
# X_test_new = pd.concat([X_test['age'], X_test_fever, X_test_cough, X_test_ohe], axis=1)
# X_train_new


#  orr you can also concatenate all numpy arrays at once and then convert it to a dataframe
X_train_age = X_train.drop(columns=['fever','city','gender','cough']).values          # .values converts the pandas DataFrame into a NumPy array
X_test_age = X_test.drop(columns=['fever','city','gender','cough']).values            # first convert age column into numpy array
        
X_train_array = np.concatenate((X_train_age, X_train_fever, X_train_cough, X_train_ohe), axis=1)         # pass all numpy array inside a tuple()
X_test_array = np.concatenate((X_test_age, X_test_fever, X_test_cough, X_test_ohe), axis=1)

# X_train_array = X_train_array.astype(int)        # convert complete numpy array into 'int'
X_train_array

array([[ 66.  ,  99.  ,   1.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 14.  , 101.  ,   1.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 10.  ,  98.  ,   1.  ,   0.  ,   0.  ,   1.  ,   0.  ],
       [ 24.  ,  98.  ,   0.  ,   1.  ,   0.  ,   1.  ,   0.  ],
       [ 80.  , 100.  ,   0.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 24.  , 102.  ,   1.  ,   0.  ,   0.  ,   0.  ,   0.  ],
       [ 14.  ,  99.  ,   0.  ,   0.  ,   0.  ,   0.  ,   1.  ],
       [ 59.  ,  99.  ,   1.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 54.  , 104.  ,   1.  ,   0.  ,   0.  ,   1.  ,   0.  ],
       [ 11.  , 100.  ,   1.  ,   0.  ,   0.  ,   1.  ,   0.  ],
       [ 33.  , 102.  ,   1.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 64.  ,  98.  ,   0.  ,   0.  ,   0.  ,   0.  ,   0.  ],
       [ 71.  , 100.92,   1.  ,   1.  ,   0.  ,   1.  ,   0.  ],
       [ 10.  , 100.  ,   0.  ,   1.  ,   0.  ,   0.  ,   0.  ],
       [ 34.  , 104.  ,   1.  ,   0.  ,   1.  ,   0.  ,   0.  ],
       [ 27.  , 100.  ,  

<h2 style="color:purple">2). Mentos Zindagi</h2>

In [ ]:
# instead of this we can simply use columnTransformer class 

In [26]:
from sklearn.compose import ColumnTransformer

In [36]:
# in transformers=  we pass the required transformation which are to be done 
# pass it in a tuple form, inside each tuple we pass 3 things : name, transformer object, and the name of columns in list[] in ehich you want to apply.
transformer = ColumnTransformer(transformers=[
                                ('tnf1',SimpleImputer(),['fever']),
                                ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']), 
                                ('tnf3',OneHotEncoder(drop='first', sparse_output=False), ['gender','city'])
                               ], 
                                remainder='passthrough'
)


# tnf1 → Imputes missing values in 'fever'
# Default strategy = mean → output will always be FLOAT

# tnf2 → Ordinal encoding for ordered categorical column
# 'Mild' → 0, 'Strong' → 1 (order matters here)

# tnf3 → One-Hot Encoding for nominal categorical columns
# drop='first' → avoids dummy variable trap
# sparse_output=False → returns NumPy array instead of sparse matrix

# remainder= 'passthrough'	      #  columns NOT mentioned above (e.g. 'age') are kept as they are
# remainder='drop'	              #  Remove remaining columns in which no transformation is applied

In [39]:
X_train_tnf = transformer.fit_transform(X_train)
X_test_tnf = transformer.transform(X_test)

display(X_train_tnf.shape)
X_train_tnf

(75, 7)

array([[ 99.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  66.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  14.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  10.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  24.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  80.        ],
       [102.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  24.        ],
       [ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  14.        ],
       [ 99.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  59.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          1.    